# Settimana 23 — Pricing avanzato e gestione di progetti IA

**Obiettivo:** costruire gli strumenti di business che userai in ogni progetto:
calcolatrice di stima PERT, generatore di preventivi in Markdown,
template di change request e calcolatrice di costo operativo per il cliente.

**Al termine avrai** in `herramientas-negocio/`:
- `estimador.py` — stima PERT per componenti
- `presupuesto.py` — generatore di preventivo in Markdown
- `change_request.py` — generatore di richieste di modifica
- `coste_operacion.py` — calcolatrice di costo mensile delle API
- Esempi d'uso con il progetto ResearchAgent come caso reale

**Costo stimato:** $0.00 — questo notebook non chiama la OpenAI API.

## Cella 0 — Setup

In [ ]:
import os, json
from pathlib import Path
from datetime import date, timedelta
from dataclasses import dataclass, field
from typing import Optional

BASE_DIR = Path("../herramientas-negocio").resolve()
BASE_DIR.mkdir(exist_ok=True)

HOY = date.today()
print(f"✓ Directory: {BASE_DIR}")
print(f"✓ Data: {HOY.isoformat()}")

## Cella 1 — Stimatore PERT

Stima ogni componente del progetto con i tre scenari e calcola il totale.

In [ ]:
@dataclass
class ComponenteProyecto:
    nombre: str
    optimista: float   # giorni
    probable: float
    pesimista: float
    descripcion: str = ""

    @property
    def pert(self) -> float:
        """Stima PERT: (O + 4P + Pe) / 6"""
        return (self.optimista + 4 * self.probable + self.pesimista) / 6

    @property
    def desviacion(self) -> float:
        """Deviazione standard PERT: (Pe - O) / 6"""
        return (self.pesimista - self.optimista) / 6


def estimar_proyecto(
    componentes: list[ComponenteProyecto],
    margen_pct: float = 0.20,
    day_rate: float = 400.0,
) -> dict:
    """
    Calcola la stima totale del progetto con margine.
    Restituisce giorni, prezzo e dettaglio per componente.
    """
    total_pert  = sum(c.pert for c in componentes)
    total_opt   = sum(c.optimista for c in componentes)
    total_pes   = sum(c.pesimista for c in componentes)
    total_con_margen = total_pert * (1 + margen_pct)

    return {
        "dias_optimista":    round(total_opt, 1),
        "dias_probable":     round(total_pert, 1),
        "dias_pesimista":    round(total_pes, 1),
        "dias_con_margen":   round(total_con_margen, 1),
        "precio_minimo_eur": round(total_pert * day_rate, 0),
        "precio_objetivo_eur": round(total_con_margen * day_rate, 0),
        "margen_aplicado":   margen_pct,
        "day_rate":          day_rate,
        "desglose": [
            {
                "nombre":    c.nombre,
                "pert_dias": round(c.pert, 1),
                "pct_total": round(c.pert / total_pert * 100, 1),
                "rango":     f"{c.optimista}-{c.pesimista}d",
            }
            for c in componentes
        ],
    }


# ── Esempio: ResearchAgent per ANFACO-CYTMA ──────────────────────────────────
COMPONENTES_RESEARCH_AGENT = [
    ComponenteProyecto(
        "Setup e infrastruttura",
        optimista=0.5, probable=1.0, pesimista=2.0,
        descripcion="FastAPI, Docker, Railway, CI/CD",
    ),
    ComponenteProyecto(
        "Ingegneria dei prompt",
        optimista=1.0, probable=3.0, pesimista=6.0,
        descripcion="Planner, executor ReAct, synthesizer",
    ),
    ComponenteProyecto(
        "Integrazione corpus cliente",
        optimista=1.0, probable=3.0, pesimista=8.0,
        descripcion="Ingestione di documenti in Chroma, embedding",
    ),
    ComponenteProyecto(
        "Sicurezza e guardrail",
        optimista=0.5, probable=1.5, pesimista=3.0,
        descripcion="Rate limiting, PII, moderazione",
    ),
    ComponenteProyecto(
        "Osservabilità",
        optimista=0.5, probable=1.0, pesimista=2.0,
        descripcion="Tracer, Langfuse, log strutturati",
    ),
    ComponenteProyecto(
        "Test e valutazione",
        optimista=1.0, probable=2.0, pesimista=4.0,
        descripcion="Test unitari, integrazione, smoke",
    ),
    ComponenteProyecto(
        "Documentazione e consegna",
        optimista=0.5, probable=1.0, pesimista=2.0,
        descripcion="README tecnico, guida d'uso, sintesi esecutiva",
    ),
]

estimacion = estimar_proyecto(
    COMPONENTES_RESEARCH_AGENT,
    margen_pct=0.25,
    day_rate=450,
)

print("STIMA PERT — ResearchAgent")
print("=" * 55)
print(f"{'Componente':<30} {'PERT':>6} {'% tot':>8} {'Range':>10}")
print("-" * 55)
for c in estimacion["desglose"]:
    print(f"{c['nombre']:<30} {c['pert_dias']:>5}d {c['pct_total']:>7}% {c['rango']:>10}")
print("-" * 55)
print(f"{'TOTALE (senza margine)':<30} {estimacion['dias_probable']:>5}d")
print(f"{'TOTALE (con margine 25%)':<30} {estimacion['dias_con_margen']:>5}d")
print()
print("PREZZO:")
print(f"  Minimo:   {estimacion['precio_minimo_eur']:,.0f}€  ({estimacion['dias_probable']}d × {estimacion['day_rate']}€/d)")
print(f"  Obiettivo: {estimacion['precio_objetivo_eur']:,.0f}€  ({estimacion['dias_con_margen']}d × {estimacion['day_rate']}€/d)")
print()
print(f"  Range da presentare al cliente: {estimacion['dias_optimista']}–{estimacion['dias_pesimista']} giorni")

## Cella 2 — Calcolatrice del costo operativo

Stima il costo mensile delle API per il cliente in tre scenari d'uso.

In [ ]:
PRECIOS_OPENAI = {
    "gpt-4o-mini": {"input": 0.15e-6,  "output": 0.60e-6},
    "gpt-4o":      {"input": 2.50e-6,  "output": 10.0e-6},
    "text-embedding-3-small": {"input": 0.02e-6, "output": 0.0},
}


def estimar_coste_mensual(
    consultas_mes: int,
    tokens_input_media: int,
    tokens_output_media: int,
    modelo: str = "gpt-4o-mini",
    llamadas_por_consulta: int = 4,  # planner + passi ReAct + synthesizer
) -> dict:
    p = PRECIOS_OPENAI[modelo]
    coste_llamada = (tokens_input_media * p["input"] +
                     tokens_output_media * p["output"])
    coste_consulta = coste_llamada * llamadas_por_consulta
    coste_mes      = coste_consulta * consultas_mes

    return {
        "modelo":              modelo,
        "consultas_mes":       consultas_mes,
        "llamadas_por_consulta": llamadas_por_consulta,
        "coste_por_consulta":  round(coste_consulta, 4),
        "coste_mes_eur":       round(coste_mes, 2),
        "coste_anio_eur":      round(coste_mes * 12, 2),
    }


# Tre scenari per il cliente
ESCENARIOS = [
    {"nombre": "Uso basso",     "consultas_mes": 50},
    {"nombre": "Uso previsto",  "consultas_mes": 200},
    {"nombre": "Uso alto",      "consultas_mes": 500},
]

# Parametri dell'agente (dai log di osservabilità della settimana 17)
TOKENS_INPUT_MEDIA  = 1_200  # per chiamata
TOKENS_OUTPUT_MEDIA = 500

print("COSTO OPERATIVO STIMATO — ResearchAgent")
print(f"Modello: gpt-4o-mini | ~{TOKENS_INPUT_MEDIA} token input + {TOKENS_OUTPUT_MEDIA} output per chiamata")
print(f"Chiamate per ricerca: 4 (planner + 2 passi ReAct + synthesizer)")
print()
print(f"{'Scenario':<16} {'Query/mese':>14} {'Costo/query':>15} {'Costo/mese':>11} {'Costo/anno':>11}")
print("-" * 70)

for esc in ESCENARIOS:
    r = estimar_coste_mensual(
        consultas_mes=esc["consultas_mes"],
        tokens_input_media=TOKENS_INPUT_MEDIA,
        tokens_output_media=TOKENS_OUTPUT_MEDIA,
    )
    print(
        f"{esc['nombre']:<16} {r['consultas_mes']:>14} "
        f"{r['coste_por_consulta']:>14.4f}€ "
        f"{r['coste_mes_eur']:>10.2f}€ "
        f"{r['coste_anio_eur']:>10.2f}€"
    )

print()
print("Nota: questi costi sono a carico del cliente (API key propria).")
print("Includi questa tabella nel contratto come 'Stima del costo operativo'.")

## Cella 3 — Generatore di preventivo in Markdown

In [ ]:
@dataclass
class DatosPresupuesto:
    # Parti
    consultor_nombre: str
    consultor_email: str
    cliente_nombre: str
    cliente_contacto: str
    # Progetto
    nombre_proyecto: str
    descripcion_breve: str
    # Stima
    componentes: list[ComponenteProyecto]
    day_rate: float
    margen_pct: float = 0.25
    # Condizioni
    dias_validez: int = 15
    anticipo_pct: float = 0.40
    modelo_precio: str = "fijo"  # "fijo" | "tm" | "retainer"
    incluye: list[str] = field(default_factory=list)
    excluye: list[str] = field(default_factory=list)
    supuestos: list[str] = field(default_factory=list)


def generar_presupuesto_md(datos: DatosPresupuesto) -> str:
    est = estimar_proyecto(datos.componentes, datos.margen_pct, datos.day_rate)
    fecha_emision = HOY.isoformat()
    fecha_validez = (HOY + timedelta(days=datos.dias_validez)).isoformat()
    precio        = est["precio_objetivo_eur"]
    anticipo      = round(precio * datos.anticipo_pct, 0)
    resto         = precio - anticipo

    include_items = "\n".join(f"- {i}" for i in datos.incluye) or "- Vedi dettaglio componenti"
    exclude_items = "\n".join(f"- {i}" for i in datos.excluye) or "- (nessuna esclusione specifica)"
    asump_items   = "\n".join(f"- {s}" for s in datos.supuestos) or "- (nessuna assunzione specifica)"

    desglose = "\n".join(
        f"| {c['nombre']:<30} | {c['rango']:>12} | {c['pert_dias']:>8}d | "
        f"{round(c['pert_dias'] * datos.day_rate):>10,}€ |"
        for c in est["desglose"]
    )

    return f"""# Preventivo — {datos.nombre_proyecto}

| | |
|---|---|
| **Consulente** | {datos.consultor_nombre} — {datos.consultor_email} |
| **Cliente** | {datos.cliente_nombre} — {datos.cliente_contacto} |
| **Data di emissione** | {fecha_emision} |
| **Valido fino al** | {fecha_validez} |
| **Numero preventivo** | PRE-{HOY.strftime('%Y%m')}-001 |

---

## Descrizione del progetto

{datos.descripcion_breve}

---

## Dettaglio dello sforzo

| Componente | Range (giorni) | Stima | Importo |
|------------|:------------:|:----------:|--------:|
{desglose}
| **SUBTOTALE** | {est['dias_optimista']}–{est['dias_pesimista']}d | **{est['dias_probable']}d** | **{est['precio_minimo_eur']:,.0f}€** |
| **Margine di contingenza ({int(datos.margen_pct*100)}%)** | | | **{precio - est['precio_minimo_eur']:,.0f}€** |
| **TOTALE** | | **{est['dias_con_margen']}d** | **{precio:,.0f}€** |

*Day rate: {datos.day_rate:,.0f}€/giorno. IVA non inclusa.*

---

## Condizioni di pagamento

| Milestone | Importo |
|------|--------:|
| Anticipo all'avvio ({int(datos.anticipo_pct*100)}%) | {anticipo:,.0f}€ |
| Consegna e accettazione ({int((1-datos.anticipo_pct)*100)}%) | {resto:,.0f}€ |
| **Totale** | **{precio:,.0f}€** |

---

## Ambito

### Incluso
{include_items}

### Non incluso
{exclude_items}

### Assunzioni del progetto
{asump_items}

Qualsiasi lavoro fuori da questo ambito richiederà una richiesta di modifica
firmata da entrambe le parti prima di iniziare.

---

## Costo operativo stimato (a carico del cliente)

| Scenario | Query/mese | Costo/mese |
|-----------|:-------------:|----------:|
| Uso basso | 50 | ~0.70€ |
| Uso previsto | 200 | ~2.80€ |
| Uso alto | 500 | ~7.00€ |

*Costo dell'API di OpenAI, con modello gpt-4o-mini. Il cliente gestisce la propria API key.*

---

*Per accettare questo preventivo, rispondi a questa email con conferma scritta
oppure firma e restituisci questo documento. L'avvio del progetto è subordinato
al versamento dell'anticipo.*
"""


# Genera il preventivo del ResearchAgent
presupuesto_datos = DatosPresupuesto(
    consultor_nombre   = "Raffaele Bini",
    consultor_email    = "raffaele@dihdatalife.com",
    cliente_nombre     = "ANFACO-CYTMA",
    cliente_contacto   = "[contacto@anfaco.es]",
    nombre_proyecto    = "ResearchAgent — Agente di ricerca automatizzata",
    descripcion_breve  = (
        "Sviluppo di un agente di ricerca autonomo che genera report strutturati "
        "in spagnolo a partire da ricerche web e dalla base documentale interna del cliente. "
        "Include API REST, guardrail di sicurezza, osservabilità e documentazione completa."
    ),
    componentes        = COMPONENTES_RESEARCH_AGENT,
    day_rate           = 450,
    margen_pct         = 0.25,
    modelo_precio      = "fijo",
    incluye=[
        "Sviluppo completo dell'agente e dell'API REST",
        "Fino a 3 iterazioni di ottimizzazione dei prompt",
        "Test automatizzati (copertura minima 80%)",
        "Documentazione tecnica, guida d'uso e sintesi esecutiva",
        "Deploy su Railway (o l'ambiente concordato)",
        "2 settimane di supporto post-consegna (risposta <24h)",
    ],
    excluye=[
        "Integrazione con sistemi non menzionati in questo preventivo",
        "Fine-tuning di modelli (richiede preventivo separato)",
        "Costo mensile dell'API di OpenAI (a carico del cliente)",
        "Manutenzione oltre il periodo di supporto post-consegna",
        "Cambi di requisiti dopo l'approvazione del design",
    ],
    supuestos=[
        "Il cliente fornisce accesso ai propri documenti entro i primi 3 giorni lavorativi",
        "È disponibile una persona di contatto tecnico per rispondere a dubbi in <24h",
        "L'ambiente di produzione ha Python 3.11+ e accesso a internet",
        "I dati di test non contengono informazioni personali sensibili di terzi",
    ],
)

presupuesto_md = generar_presupuesto_md(presupuesto_datos)
(BASE_DIR / "presupuesto_research_agent.md").write_text(presupuesto_md)
print("✓ presupuesto_research_agent.md generato")
print()
# Mostra il riepilogo del prezzo
est = estimar_proyecto(COMPONENTES_RESEARCH_AGENT, 0.25, 450)
print(f"Prezzo da presentare: {est['precio_objetivo_eur']:,.0f}€")
print(f"Anticipo (40%):       {est['precio_objetivo_eur']*0.4:,.0f}€")
print(f"Resto alla consegna:  {est['precio_objetivo_eur']*0.6:,.0f}€")

## Cella 4 — Generatore di change request

In [ ]:
@dataclass
class ChangeRequest:
    numero: int
    proyecto: str
    solicitado_por: str
    descripcion: str
    justificacion: str
    opciones: list[dict]  # [{"nombre", "dias", "precio_eur", "impacto_plazo"}]
    fecha: str = ""

    def __post_init__(self):
        if not self.fecha:
            self.fecha = HOY.isoformat()


def generar_change_request_md(cr: ChangeRequest) -> str:
    opciones_txt = ""
    for i, op in enumerate(cr.opciones, 1):
        opciones_txt += f"""
### Opzione {i}: {op['nombre']}

- **Sforzo stimato:** {op['dias']} giorni
- **Costo aggiuntivo:** {op['precio_eur']:,.0f}€ + IVA
- **Impatto sui tempi:** {op['impacto_plazo']}
- **Descrizione:** {op.get('descripcion', '')}
"""

    return f"""# Richiesta di modifica #{cr.numero:03d}

| | |
|---|---|
| **Progetto** | {cr.proyecto} |
| **Richiesto da** | {cr.solicitado_por} |
| **Data** | {cr.fecha} |
| **Stato** | ⏳ In attesa di approvazione |

---

## Modifica richiesta

{cr.descripcion}

## Motivazione

{cr.justificacion}

---

## Opzioni proposte
{opciones_txt}

---

## Decisione

- [ ] **Opzione approvata:** _______________
- [ ] **Respinta** — Motivo: _______________
- [ ] **Rinviata alla fase 2**

**Firma del cliente:** _______________ **Data:** _______________

**Firma del consulente:** _______________ **Data:** _______________

*Questo documento fa parte del contratto del progetto. Nessun lavoro aggiuntivo
inizierà senza la firma di entrambe le parti.*
"""


# Esempio reale: il cliente chiede di aggiungere l'integrazione con EUMOFA
cr_ejemplo = ChangeRequest(
    numero=1,
    proyecto="ResearchAgent — ANFACO-CYTMA",
    solicitado_por="Direttore Innovazione, ANFACO-CYTMA",
    descripcion=(
        "Aggiungere il supporto per consultare i prezzi di mercato del database EUMOFA "
        "(European Market Observatory for Fisheries and Aquaculture) direttamente dall'"
        "agente di ricerca."
    ),
    justificacion=(
        "I report attuali non includono dati sui prezzi di mercato, che sono fondamentali "
        "per le analisi di competitività del settore conserviero."
    ),
    opciones=[
        {
            "nombre": "Integrazione base (scraping web EUMOFA)",
            "dias": 3,
            "precio_eur": 1350,
            "impacto_plazo": "+3 giorni lavorativi",
            "descripcion": "Scraping del portale pubblico di EUMOFA. Fragile rispetto a cambiamenti del sito.",
        },
        {
            "nombre": "Integrazione API ufficiale EUMOFA",
            "dias": 5,
            "precio_eur": 2250,
            "impacto_plazo": "+1 settimana",
            "descripcion": "Uso dell'API REST ufficiale. Richiede registrazione e chiave di accesso (gratuita).",
        },
        {
            "nombre": "Rinviare alla fase 2",
            "dias": 0,
            "precio_eur": 0,
            "impacto_plazo": "Nessun impatto sul progetto attuale",
            "descripcion": "Includere in uno sprint di miglioramenti dopo la consegna. Preventivo separato.",
        },
    ],
)

cr_md = generar_change_request_md(cr_ejemplo)
(BASE_DIR / "change_request_001.md").write_text(cr_md)
print("✓ change_request_001.md generato")
print()
print(cr_md[:800], "...")

## Cella 5 — Rilevatore di segnali d'allarme nel progetto

In [ ]:
@dataclass
class EstadoProyecto:
    dias_desde_inicio: int
    dias_estimados_total: float
    dias_trabajados: float
    pct_entregado: float       # 0.0 - 1.0
    dias_sin_datos_cliente: int
    num_cambios_requisitos: int
    reuniones_sin_decisor: int
    scope_creep_dias: float    # giorni di lavoro fuori ambito già svolti


def analizar_salud_proyecto(estado: EstadoProyecto) -> dict:
    alertas = []
    advertencias = []

    # Velocità di avanzamento
    velocidad_esperada = estado.dias_desde_inicio / estado.dias_estimados_total
    if estado.pct_entregado < velocidad_esperada * 0.7:
        alertas.append(
            f"Ritmo di consegna basso: {estado.pct_entregado*100:.0f}% completato "
            f"con il {velocidad_esperada*100:.0f}% del tempo consumato."
        )

    # Blocchi del cliente
    if estado.dias_sin_datos_cliente > 3:
        alertas.append(
            f"Blocco da parte del cliente: {estado.dias_sin_datos_cliente} giorni senza ricevere "
            f"dati o accessi necessari. Notifica e documenta."
        )

    # Cambi di requisiti
    if estado.num_cambios_requisitos >= 2:
        alertas.append(
            f"{estado.num_cambios_requisitos} cambi di requisiti. "
            f"Ogni cambio senza change request firmata è un rischio contrattuale."
        )
    elif estado.num_cambios_requisitos == 1:
        advertencias.append("1 cambio di requisiti. Hai la change request firmata?")

    # Riunioni senza decisore
    if estado.reuniones_sin_decisor >= 2:
        advertencias.append(
            f"{estado.reuniones_sin_decisor} riunioni senza il decisore finale. "
            f"Chiedi una riunione di allineamento con chi approva."
        )

    # Scope creep
    pct_scope_creep = estado.scope_creep_dias / estado.dias_estimados_total
    if pct_scope_creep > 0.15:
        alertas.append(
            f"Scope creep: {estado.scope_creep_dias}d di lavoro fuori ambito "
            f"({pct_scope_creep*100:.0f}% del totale). Emetti immediatamente una change request."
        )
    elif pct_scope_creep > 0.05:
        advertencias.append(
            f"{estado.scope_creep_dias}d di lavoro fuori ambito "
            f"({pct_scope_creep*100:.0f}%). Monitora da vicino."
        )

    nivel = "🔴 CRITICO" if alertas else ("🟡 ATTENZIONE" if advertencias else "🟢 OK")

    return {
        "nivel":       nivel,
        "alertas":     alertas,
        "advertencias": advertencias,
        "resumen":     f"{len(alertas)} alert, {len(advertencias)} avviso/i",
    }


# Esempio: progetto con alcuni problemi
ESCENARIOS_PROYECTO = [
    {
        "nombre": "Progetto sano",
        "estado": EstadoProyecto(
            dias_desde_inicio=5, dias_estimados_total=14,
            dias_trabajados=5, pct_entregado=0.38,
            dias_sin_datos_cliente=0, num_cambios_requisitos=0,
            reuniones_sin_decisor=0, scope_creep_dias=0,
        )
    },
    {
        "nombre": "Progetto con scope creep",
        "estado": EstadoProyecto(
            dias_desde_inicio=8, dias_estimados_total=14,
            dias_trabajados=8, pct_entregado=0.40,
            dias_sin_datos_cliente=2, num_cambios_requisitos=2,
            reuniones_sin_decisor=1, scope_creep_dias=2.5,
        )
    },
    {
        "nombre": "Progetto a rischio",
        "estado": EstadoProyecto(
            dias_desde_inicio=12, dias_estimados_total=14,
            dias_trabajados=12, pct_entregado=0.50,
            dias_sin_datos_cliente=5, num_cambios_requisitos=3,
            reuniones_sin_decisor=3, scope_creep_dias=4,
        )
    },
]

for esc in ESCENARIOS_PROYECTO:
    resultado = analizar_salud_proyecto(esc["estado"])
    print(f"{'='*55}")
    print(f"{resultado['nivel']}  {esc['nombre']}")
    print(f"  {resultado['resumen']}")
    for a in resultado["alertas"]:
        print(f"  ⚠ ALERT: {a}")
    for w in resultado["advertencias"]:
        print(f"  ℹ  AVVISO:  {w}")
    print()

## Cella 6 — Salvataggio degli strumenti come moduli riutilizzabili

In [ ]:
# Elenca tutti i deliverable generati
print("Deliverable generati in herramientas-negocio/:")
print()
for f in sorted(BASE_DIR.rglob("*.md")):
    rel = f.relative_to(BASE_DIR)
    lineas = len(f.read_text().splitlines())
    print(f"  {rel}  ({lineas} righe)")

print()
print("Funzioni disponibili in questo notebook:")
funciones = [
    ("estimar_proyecto(componentes, margen, day_rate)",
     "Stima PERT totale con dettaglio per componente"),
    ("estimar_coste_mensual(consultas, tokens_in, tokens_out, modelo)",
     "Costo mensile delle API per il cliente"),
    ("generar_presupuesto_md(DatosPresupuesto)",
     "Preventivo completo in Markdown pronto per l'invio"),
    ("generar_change_request_md(ChangeRequest)",
     "Richiesta di modifica con opzioni e firma"),
    ("analizar_salud_proyecto(EstadoProyecto)",
     "Rileva scope creep, blocchi e alert"),
]
for nombre, desc in funciones:
    print(f"  {nombre}")
    print(f"    → {desc}")
    print()

## Sfida — Simulatore di negoziazione del prezzo

Il cliente ti dice che il preventivo che gli hai inviato è del 30% superiore a quanto previsto.
Invece di abbassare direttamente il prezzo, la tecnica corretta è aggiustare l'ambito.

Implementa `proponer_alcance_reducido(componentes, presupuesto_cliente)` che:

1. Calcola quanti giorni di lavoro rientrano nel budget del cliente (con il tuo day rate)
2. Ordina i componenti per costo dal maggiore al minore
3. Propone due versioni ridotte del progetto:
   - **MVP**: elimina i componenti a priorità più bassa finché non rientra nel budget
   - **Fase 1 / Fase 2**: divide i componenti in due consegne, la prima entro il budget
4. Genera un Markdown con le due proposte da inviare al cliente

```python
def proponer_alcance_reducido(
    componentes: list[ComponenteProyecto],
    presupuesto_cliente: float,
    day_rate: float,
    prioridades: list[str],  # nomi dei componenti in ordine di priorità
) -> str:
    # TODO
    pass

# Prova: il cliente può pagare solo 4.000€
propuesta = proponer_alcance_reducido(
    COMPONENTES_RESEARCH_AGENT,
    presupuesto_cliente=4000,
    day_rate=450,
    prioridades=[
        "Setup e infrastruttura",
        "Ingegneria dei prompt",
        "Test e valutazione",
        "Integrazione corpus cliente",
        "Documentazione e consegna",
        "Sicurezza e guardrail",
        "Osservabilità",
    ],
)
print(propuesta)
```

## Riepilogo della settimana 23

| Strumento | Dataclass / funzione | Quando usarlo |
|------------|--------------------|--------------|
| Stima PERT | `ComponenteProyecto` + `estimar_proyecto()` | Prima di dare qualsiasi prezzo |
| Costo operativo | `estimar_coste_mensual()` | Nel contratto, sezione 'costi a carico del cliente' |
| Preventivo | `DatosPresupuesto` + `generar_presupuesto_md()` | Da inviare al cliente prima della firma |
| Change request | `ChangeRequest` + `generar_change_request_md()` | Ogni volta che il cliente chiede qualcosa fuori ambito |
| Salute del progetto | `EstadoProyecto` + `analizar_salud_proyecto()` | Revisione settimanale durante il progetto |

La settimana 24 è la **settimana di chiusura del piano**: revisione del portfolio,
piano d'azione per ottenere il primo cliente e roadmap dei prossimi sei mesi.